In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, subprocess, shutil
import numpy as np, pandas as pd
import torch
from transformers import AutoModel
import soundfile as sf

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_checkpoint.json'
TEMP_DIR = '/tmp/audio_wav'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

# Build audio lookup
audio_exts = ('.m4a', '.wav', '.mp3', '.webm')
available_audio = {}
if os.path.exists(AUDIO_DIR):
    for f in os.listdir(AUDIO_DIR):
        if not any(f.endswith(e) for e in audio_exts):
            continue
        vid_base = f.rsplit('.', 1)[0]
        if vid_base not in available_audio:
            available_audio[vid_base] = os.path.join(AUDIO_DIR, f)
        if ',' in vid_base:
            vid_clean = vid_base.split(',')[0]
            if vid_clean not in available_audio:
                available_audio[vid_clean] = os.path.join(AUDIO_DIR, f)

available_labels = set()
if os.path.exists(LABEL_DIR):
    for f in os.listdir(LABEL_DIR):
        if f.endswith('.csv'):
            available_labels.add(f.replace('.csv', ''))

overlap = sorted(available_audio.keys() & available_labels - done)
print(f'Audio: {len(available_audio)} | Labels: {len(available_labels)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')
if overlap:
    print(f'Sample: {overlap[:3]}')


In [ ]:
# Load WavLM on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Loading WavLM-base...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM loaded!')


In [ ]:
# Per-word feature extraction (with ffmpeg fallback)
SR = 16000
MIN_SAMPLES = int(0.02 * SR)  # 20ms minimum

def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        p = ts_str.strip('[]').split(',')
        return float(p[0]), float(p[1])
    except:
        return None, None

def load_segment(audio_path, t0, t1):
    """Load audio segment [t0, t1] seconds as mono float32 at SR Hz.
    
    Strategy:
    1. soundfile with start/stop (fast, exact slice)
    2. If soundfile fails, ffmpeg convert to wav in /tmp, load from there
    3. If ffmpeg fails, skip (return zeros)
    """
    dur = t1 - t0
    if dur <= 0:
        return np.zeros(MIN_SAMPLES, dtype=np.float32)
    
    # Try soundfile first
    try:
        y, sr = sf.read(audio_path, start=int(t0 * SR),
                       stop=int(t1 * SR), dtype='float32')
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        if len(y) < MIN_SAMPLES:
            y = np.pad(y, (0, MIN_SAMPLES - len(y)))
        return y.astype(np.float32)
    except Exception as e1:
        pass
    
    # Try ffmpeg convert to wav and load
    try:
        vid = os.path.basename(audio_path).rsplit('.', 1)[0]
        wav_path = os.path.join(TEMP_DIR, vid + '.wav')
        
        # Only convert if not already converted
        if not os.path.exists(wav_path):
            cmd = [
                'ffmpeg', '-y', '-i', audio_path,
                '-ar', str(SR), '-ac', '1', '-acodec', 'pcm_s16le',
                '-ss', str(t0), '-t', str(dur),
                wav_path
            ]
            r = subprocess.run(cmd, capture_output=True, timeout=30)
        
        if os.path.exists(wav_path):
            y, _ = sf.read(wav_path, dtype='float32')
            if len(y.shape) > 1:
                y = y.mean(axis=1)
            if len(y) < MIN_SAMPLES:
                y = np.pad(y, (0, MIN_SAMPLES - len(y)))
            return y.astype(np.float32)
    except Exception as e2:
        pass
    
    # Last resort: zeros
    return np.zeros(MIN_SAMPLES, dtype=np.float32)

def extract_word_features(audio_path, word_times, batch_size=32):
    n_words = len(word_times)
    if n_words == 0:
        return None
    
    features = []
    for batch_start in range(0, n_words, batch_size):
        batch_times = word_times[batch_start:batch_start + batch_size]
        segments, valid_mask = [], []
        
        for t0, t1 in batch_times:
            seg = load_segment(audio_path, t0, t1)
            valid = (t1 > t0)
            segments.append(seg)
            valid_mask.append(valid)
        
        if not segments:
            continue
        
        # Pad to same length
        max_len = max(len(s) for s in segments)
        padded = []
        for s in segments:
            if len(s) < max_len:
                s = np.pad(s, (0, max_len - len(s)))
            padded.append(s)
        
        batch = torch.tensor(np.stack(padded), dtype=torch.float32).to(device)
        with torch.no_grad():
            out = wavlm(batch).last_hidden_state  # (batch, seq, 768)
            emb = out.mean(dim=1).squeeze(1)       # (batch, 768)
        
        for i, v in enumerate(valid_mask):
            if v:
                features.append(emb[i].cpu().numpy())
            else:
                features.append(np.zeros(768, dtype=np.float32))
    
    return np.array(features, dtype=np.float32)

print('Feature extractor ready')


In [ ]:
# Process all videos
SKIP_THRESHOLD = 50

t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = os.path.join(OUT_DIR, vid + '_word_features.npy')
    if os.path.exists(out_file):
        continue
    
    audio_path = available_audio[vid]
    label_path = os.path.join(LABEL_DIR, vid + '.csv')
    
    df = pd.read_csv(label_path)
    
    # Get aligned timestamps and labels
    word_times, word_labels = [], []
    for _, row in df.iterrows():
        t0_w, t1_w = parse_timestamp(row['timestamp'])
        if t0_w is not None:
            word_times.append((t0_w, t1_w))
            word_labels.append(str(row['label']).strip())
    
    if len(word_times) < SKIP_THRESHOLD:
        done.add(vid)
        continue
    
    try:
        feats = extract_word_features(audio_path, word_times)
    except Exception as e:
        print(f'ERROR {vid}: {e}')
        continue
    
    if feats is None or len(feats) == 0:
        continue
    
    assert len(feats) == len(word_labels), f'{vid}: {len(feats)} != {len(word_labels)}'
    
    np.save(out_file, feats)
    done.add(vid)
    
    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape} | done={len(done)} | rate={rate:.0f}/hr')
    
    if len(done) % 10 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)

print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
# Summary
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(os.path.join(OUT_DIR, f))
    print(f'  {f}: {d.shape}')
